In [ ]:
!pip install pandas numpy scikit-learn joblib matplotlib imbalanced-learn

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib
import json

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

print("Loading data...")
df = pd.read_csv('data.csv')

Loading data...


In [3]:
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [ ]:


# Prepare Target
df['Attrition'] = df['Attrition'].apply(lambda x: 1 if x == 'Yes' else 0)
y = df['Attrition']

# Drop useless columns
drop_cols = ['Attrition', 'EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
X = df.drop(columns=drop_cols)

# Column routing
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])


pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42)) 
   
])

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training the Random Forest Pipeline with SMOTE...")
pipeline.fit(X_train, y_train)

# Evaluate
y_pred = pipeline.predict(X_test)
print("\n--- Model Evaluation ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Extract Feature Importance
rf_model = pipeline.named_steps['classifier']
encoded_cat_features = pipeline.named_steps['preprocessor'].transformers_[1][1].get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(encoded_cat_features)

importances = rf_model.feature_importances_
feature_importance_dict = sorted(zip(all_feature_names, importances), key=lambda x: x[1], reverse=True)

top_features = [{"feature": k, "importance": float(v)} for k, v in feature_importance_dict[:10]]
with open('feature_importances.json', 'w') as f:
    json.dump(top_features, f)

joblib.dump(pipeline, 'hr_pipeline.joblib')
print("\nSuccess! Pipeline saved as 'hr_pipeline.joblib'")

Loading data...
Training the Random Forest Pipeline with SMOTE...

--- Model Evaluation ---
Accuracy: 0.84

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.96      0.91       247
           1       0.52      0.26      0.34        47

    accuracy                           0.84       294
   macro avg       0.70      0.61      0.63       294
weighted avg       0.82      0.84      0.82       294


Success! Pipeline saved as 'hr_pipeline.joblib'
